In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StructField, DoubleType
from pyspark.ml.feature import VectorAssembler
import joblib
import numpy as np

# -------------------------
# 1️⃣ Spark session
# -------------------------
spark = SparkSession.builder \
    .appName("KafkaSparkXGB_CountFraud") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

# -------------------------
# 2️⃣ Schema
# -------------------------
schema = StructType(
    [StructField("Time", DoubleType(), True)] +
    [StructField(f"V{i}", DoubleType(), True) for i in range(1, 29)] +
    [StructField("Amount", DoubleType(), True)]
)

# -------------------------
# 3️⃣ Kafka stream
# -------------------------
kafka_df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", "topic1") \
    .option("startingOffsets", "earliest") \
    .load()

# -------------------------
# 4️⃣ Parse JSON
# -------------------------
parsed_df = kafka_df.select(
    from_json(col("value").cast("string"), schema).alias("data")
).select("data.*")

# -------------------------
# 5️⃣ Feature Engineering
# -------------------------
feature_cols = ["Time"] + [f"V{i}" for i in range(1, 29)] + ["Amount"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
features_df = assembler.transform(parsed_df)

# -------------------------
# 6️⃣ Load model
# -------------------------
xgb_model = joblib.load("../../models/xgboost_fraud_model2.pkl")

# -------------------------
# 7️⃣ Global counters
# -------------------------
total_seen = 0
total_predicted_fraud = 0

# -------------------------
# 8️⃣ Prediction + counting
# -------------------------
def predict_and_count(batch_df, batch_id):
    global total_seen, total_predicted_fraud

    batch_count = batch_df.count()
    if batch_count == 0:
        return

    pdf = batch_df.select("features").toPandas()
    X = np.array(pdf["features"].tolist())

    probas = xgb_model.predict_proba(X)[:, 1]

    # ⚠️ Choose threshold carefully
    predictions = (probas > 0.02).astype(int)

    batch_fraud = predictions.sum()

    total_seen += batch_count
    total_predicted_fraud += batch_fraud

    pdf["fraud_proba"] = probas
    pdf["prediction"] = predictions
    pdf["fraud"] = np.where(predictions == 1, "FRAUD", "")

    print("-------------------------------------------------------")

    print(f"\n========== BATCH {batch_id} ==========")

    print(pdf.head(5))

    print(f"Rows in batch: {batch_count}") # combien de transactions  dans le batch
    print(f"Fraud predicted in batch: {batch_fraud}") # fraud  dans le batch
    print(f"TOTAL rows seen: {total_seen}") # total transactions
    print(f"TOTAL fraud predicted: {total_predicted_fraud}") # total fraud 

    print("--------------------------------------------------------")

# -------------------------
# 9️⃣ Start stream
# -------------------------
query = features_df.writeStream \
    .foreachBatch(predict_and_count) \
    .trigger(processingTime="1 second") \
    .start()

query.awaitTermination()


26/01/05 20:34:38 WARN Utils: Your hostname, DESKTOP-DD5N9SQ resolves to a loopback address: 127.0.1.1; using 172.17.138.117 instead (on interface eth0)
26/01/05 20:34:38 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/souhail/.local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/souhail/.ivy2/cache
The jars for the packages stored in: /home/souhail/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-fdec2a87-9405-47cf-b1f7-2fab2dd66c0b;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.4.1 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.4.1 in central
	found org.apache.kafka#kafka-clients;3.3.2 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.1 in central
	found org.slf4j#slf4j-api;2.0.6 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
:: resolution report :: resolve 2722ms :: artifacts dl 75ms


-------------------------------------------------------

========== BATCH 0 ==========
                                            features  fraud_proba  prediction  \
0  [10.0, 1.249998742053, -1.22163680921816, 0.38...     0.000015           0   
1  [23.0, 1.17328461017079, 0.353497876646456, 0....     0.000029           0   
2  [25.0, 1.11400859541157, 0.0855460896870318, 0...     0.000110           0   
3  [35.0, 1.38639697419213, -0.794209465382612, 0...     0.000038           0   
4  [44.0, -0.7147640190509, 0.514968588809346, 1....     0.000034           0   

  fraud  
0        
1        
2        
3        
4        
Rows in batch: 375
Fraud predicted in batch: 2
TOTAL rows seen: 375
TOTAL fraud predicted: 2
--------------------------------------------------------
-------------------------------------------------------

========== BATCH 1 ==========
                                            features  fraud_proba  prediction  \
0  [4989.0, 0.0830133938704318, -2.5757588514435

-------------------------------------------------------

========== BATCH 5 ==========
                                            features  fraud_proba  prediction  \
0  [8770.0, -2.51028652876069, 1.8476875679587, 1...     0.000041           0   
1  [8786.0, -1.22997384428245, 1.40144325107071, ...     0.000047           0   
2  [8962.0, -1.31318176945486, 2.43390513793495, ...     0.000034           0   
3  [9005.0, -1.21712910582858, 1.81509229368554, ...     0.000049           0   
4  [9019.0, -1.24140351222608, -0.290988308095852...     0.000004           0   

  fraud  
0        
1        
2        
3        
4        
Rows in batch: 18
Fraud predicted in batch: 0
TOTAL rows seen: 531
TOTAL fraud predicted: 4
--------------------------------------------------------
-------------------------------------------------------

========== BATCH 6 ==========
                                            features  fraud_proba  prediction  \
0  [9317.0, -2.12373651817835, 1.06044419630134, 

...

-------------------------------------------------------

========== BATCH 805 ==========
                                            features  fraud_proba  prediction  \
0  [68895.0, -0.236061783040538, 0.00519067661287...     0.000406           0   
1  [68899.0, 1.33217785112586, -0.593349214105578...     0.000041           0   
2  [68902.0, 1.1630396914914, -0.788172076731251,...     0.000070           0   
3  [68903.0, -0.721755091740849, 0.72749924954031...     0.000031           0   
4  [68912.0, 1.19836827605746, 0.275895193979132,...     0.000170           0   

  fraud  
0        
1        
2        
3        
4        
Rows in batch: 10
Fraud predicted in batch: 0
TOTAL rows seen: 8272
TOTAL fraud predicted: 95
--------------------------------------------------------
-------------------------------------------------------

========== BATCH 806 ==========
                                            features  fraud_proba  prediction  \
0  [68945.0, 1.06751383928524, -0.194951940

ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/home/souhail/.local/lib/python3.10/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
RuntimeError: reentrant call inside <_io.BufferedReader name=59>

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/souhail/.local/lib/python3.10/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/home/souhail/.local/lib/python3.10/site-packages/py4j/clientserver.py", line 539, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving
ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/home/souhail/.local/lib/python3.10/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline(

-------------------------------------------------------

========== BATCH 1372 ==========
                                            features  fraud_proba  prediction  \
0  [121936.0, 1.88913777015443, -0.10451833905614...     0.000129           0   
1  [121938.0, 2.05834744648961, 0.073323538866932...     0.000155           0   
2  [121943.0, 1.80707712543234, -0.98108265401240...     0.000088           0   
3  [121963.0, -0.847440207548215, 0.0666811672163...     0.000023           0   
4  [121970.0, 2.22730184837176, -0.73080165591498...     0.000047           0   

  fraud  
0        
1        
2        
3        
4        
Rows in batch: 12
Fraud predicted in batch: 0
TOTAL rows seen: 13727
TOTAL fraud predicted: 147
--------------------------------------------------------
-------------------------------------------------------

========== BATCH 1373 ==========
                                            features  fraud_proba  prediction  \
0  [122001.0, 2.07443282609369, 0.06806

Py4JError: An error occurred while calling o67.awaitTermination

-------------------------------------------------------

========== BATCH 1376 ==========
                                            features  fraud_proba  prediction  \
0  [122139.0, -1.19752364326814, -0.0269463385528...     0.000061           0   
1  [122150.0, -0.54530989968468, 2.21377812510744...     0.000171           0   
2  [122150.0, 0.697563903796105, -2.6043306911539...     0.000160           0   
3  [122153.0, 2.03499679478385, 0.273057863689555...     0.000245           0   
4  [122156.0, 1.84860087316215, -0.64563032928484...     0.000195           0   

  fraud  
0        
1        
2        
3        
4        
Rows in batch: 28
Fraud predicted in batch: 0
TOTAL rows seen: 13782
TOTAL fraud predicted: 148
--------------------------------------------------------
-------------------------------------------------------

========== BATCH 1377 ==========
                                            features  fraud_proba  prediction  \
0  [122253.0, -0.0759593964958621, -0.4